Staging: StatCan trade by NACIS classification
==============================================

In [1]:
import duckdb as dd
from time import sleep
from pathlib import Path 
import pandas as pd 
import sys

project_root = Path.cwd().parent.parent.parent
sys.path.insert(0, str(project_root))

from utils.paths import (RAW_STATCAN_DIR,
                         STG_STATCAN_DIR)

In [2]:
TRADE_TABLE_ID = 12100176

trade_by_naics_filename = f'raw_statcan__tbl_{str(TRADE_TABLE_ID)}.parquet'
trade_by_naics_filepath = RAW_STATCAN_DIR / trade_by_naics_filename

In [3]:
# Query parameters
trade_by_naics_path = str(trade_by_naics_filepath)
imports = 'Imports'
exports = 'Exports'
industry_grp_regexp = r'\[(\d{4})\]$'

In [4]:
con = dd.connect()

In [5]:
stg_query = f"""
-- This query cleans and prepares the raw trade data.
-- 1. Selects only the necessary columns.
-- 2. Renames columns to a consistent snake_case format.
-- 3. Extracts the year from the REF_DATE string.
-- 4. Filters for the relevant trade types and date range.

SELECT
    CAST(LEFT(REF_DATE, 4) AS INT) AS year,
    Trade AS trade_type,
    "Trading Partners" AS trading_partner,
    "North American Industry Classification System (NAICS)" AS naics,
    VALUE AS cad_value
FROM
    read_parquet('{trade_by_naics_filepath}')
WHERE
    trade_type IN ('{imports}', '{exports}')
    -- Filter for either the total or the 4-digit NAICS groups we want to rank
    AND (
        naics = 'Total of all industries'
        OR REGEXP_MATCHES(naics, '{industry_grp_regexp}')
    )
"""

stg_trade_by_naics = 'stg_statcan__trade_by_naics'
con.sql(f"CREATE OR REPLACE VIEW '{stg_trade_by_naics}' AS ({stg_query})")  # Optional

In [6]:
out_filename = f'{stg_trade_by_naics}.parquet'
out_filepath = STG_STATCAN_DIR / out_filename

save_query = f"""
COPY
    (SELECT * FROM '{stg_trade_by_naics}')
TO
    '{out_filepath.as_posix()}'
(FORMAT PARQUET);
"""

con.execute(save_query)
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))